In [4]:
import torch
import pandas as pd
from torch.utils.data import DataLoader, Dataset, random_split
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np

In [5]:
###########################################
#         Normalize Transform             #
###########################################
# ADDED: Define a normalization transform class.
class Normalize:
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def __call__(self, x):
        return (x - self.mean) / self.std
    
###########################################
#       Custom Dataset Definition       #
###########################################
class CustomDataset(Dataset):
    def __init__(self, class0_csv, class1_csv, transform=None):
        """
        read csv files of 2 class. it is assumed that they dont have headers and labels 
        and all features are numerical
        """
        super(CustomDataset, self).__init__()
        data0 = pd.read_csv(class0_csv,header=None)
        data1 = pd.read_csv(class1_csv, header=None)
        data0["Label"]=0
        data1["Label"]=1
        data = pd.concat([data0, data1], ignore_index=True)
        data = data.sample(frac=1, random_state=123)
        
        self.X = data.drop(columns=["Label"]).values
        self.y = data["Label"].values
        self.transform = transform
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, index):
        X = self.X[index]
        y = self.y[index]
        
        if self.transform:
            X = self.transform(X)
        
        return torch.tensor(X).to(torch.float32), torch.tensor(y).to(torch.float32)
    
###########################################
#         Logistic Regression Model       #
###########################################
class LogesticRegression(nn.Module):
    def __init__(self, input_dim):
        #binary logestic regression classification with one linear layer
        super(LogesticRegression, self).__init__()
        self.linear = nn.Linear(input_dim, 1)
        
    def forward(self, X):
        logits = self.linear(X)
        prob = torch.sigmoid(logits)

        return prob.flatten()
    
        


In [6]:
###########################################
#                Training                 #
###########################################
def train_model(model, epochs, loader, optimizer, criterion, device):
    model.train()
    for ep in range(epochs):
        running_loss = 0
        for input, target in loader:
            input, target = input.to(device), target.to(device)
            
            optimizer.zero_grad()
            
            output = model(input)
            loss = criterion(output, target)
            
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * input.size(0)
        #mean loss of this epoch
        epoch_loss = running_loss/len(loader.dataset)
        if (ep+1) % 10 == 0 or ep == 0:
            print("loss for epoch: ", ep+1, "/", epochs, "is equal to:", epoch_loss)
            torch.save(model.state_dict(), f'model_epoch_{ep+1}.pth')
            
###########################################
#               Evaluating                #
###########################################
def eval_model(model, loader, device):
    model.eval()
    all_targets, all_predicts = [], []
    with torch.no_grad():
        for input, target in loader:
            input, target = input.to(device), target.to(device)
            preds = model(input)
            #transforming from probablities to classes
            preds = (preds >= 0.5) * 1
            all_predicts.append(preds.cpu())
            all_targets.append(target.cpu())
    # transforming to numpy and then faltten it (from 2d to 1d)    

    all_predicts_np = torch.tensor(all_predicts).numpy().flatten()
    all_targets_np = torch.tensor(all_targets).numpy().flatten()
    
    acc = accuracy_score(all_targets_np, all_predicts_np)
    print(f"\nTest Accuracy: {acc:.2f}")
    print("\nClassification Report:")
    print(classification_report(all_targets_np, all_predicts_np, digits=4))
    print("Confusion Matrix:")
    print(confusion_matrix(all_targets_np, all_predicts_np))

        

In [7]:
###########################################
#              Main Training              #
###########################################
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)
    
    class0_data = "../../dataset/4096/nature_RAR.csv"
    class1_data = "../../dataset/4096/nature_ENCRAR.csv"
    dataset = CustomDataset(class0_data, class1_data)
    input_dim = dataset.X.shape[1]
    train_size = int(0.8*len(dataset))
    test_size = len(dataset) - train_size
    train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
    
    batch_size = 64
    epochs = 100
    learning_rate = 0.01
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size)
    test_loader = DataLoader(test_dataset)
    
    # ADDED: Compute normalization parameters from the training data.
    train_indices = train_dataset.indices  # Indices of training samples.
    train_samples = dataset.X[train_indices]
    norm_mean = train_samples.mean(axis=0)
    norm_std = train_samples.std(axis=0)
    
    # ADDED: Create a normalization transform using the training parameters.
    normalize_transform = Normalize(norm_mean, norm_std)
    
    # ADDED: Assign the normalization transform to the underlying dataset.
    # Because both train_dataset and test_dataset reference the same dataset object,
    # both will apply the normalization transform in __getitem__.
    dataset.transform = normalize_transform
    
    model = LogesticRegression(input_dim).to(device)
    
    criterion = torch.nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    
    print("\n start training \n")
    train_model(model=model, epochs=epochs, loader=train_loader, optimizer=optimizer, criterion=criterion, device=device)
    print("\n end of training \n")

Using device: cuda

 start training 

loss for epoch:  1 / 100 is equal to: 1.8740562302047723
loss for epoch:  10 / 100 is equal to: 1.89304545899095
loss for epoch:  20 / 100 is equal to: 1.893045405678326
loss for epoch:  30 / 100 is equal to: 1.893045378909298
loss for epoch:  40 / 100 is equal to: 1.8930454124718359
loss for epoch:  50 / 100 is equal to: 1.8930454188507813
loss for epoch:  60 / 100 is equal to: 1.8930454328867539
loss for epoch:  70 / 100 is equal to: 1.8930453946818568
loss for epoch:  80 / 100 is equal to: 1.8930454032635349
loss for epoch:  90 / 100 is equal to: 1.893045443489673
loss for epoch:  100 / 100 is equal to: 1.8930453737702255

 end of training 



In [9]:
###########################################
#              Main Testing               #
###########################################

if __name__ == "__main__":
    model = LogesticRegression(input_dim=input_dim).to(device)
    ## Load the state dictionary
    state_dict = torch.load("./model_epoch_10.pth")

    ## Load state dictionary into the model
    model.load_state_dict(state_dict)

    # Set to evaluation mode
    
    eval_model(model=model, loader=test_loader, device=device)


Test Accuracy: 0.51

Classification Report:
              precision    recall  f1-score   support

         0.0     0.5165    0.5120    0.5143     15667
         1.0     0.5121    0.5166    0.5143     15533

    accuracy                         0.5143     31200
   macro avg     0.5143    0.5143    0.5143     31200
weighted avg     0.5143    0.5143    0.5143     31200

Confusion Matrix:
[[8022 7645]
 [7509 8024]]
